In [1]:
import os, re
import numpy as np
from scipy.io import wavfile
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# -------------------------
# CONFIG
# -------------------------
CUDA_DEVICE = 0
TORCH_DTYPE = torch.float16
TRANSCRIPT_MODEL_ID = "models/whisper-large-v3-turbo"   # your local path or HF id
FILE_STEM = "voice-1"                 # without extension, expects FILE_STEM.wav
CHUNK_SECONDS = 300                                    # 5 min chunks
LANGUAGE = None                                        # e.g. "en" or None to auto
MAX_CHARS_PER_LINE = 32                                # IG-friendly
DEFAULT_DUR = 1.2                                      # caption duration if end missing
MIN_GAP = 0.04                                         # small gap between captions

# -------------------------
# DEVICE SETUP
# -------------------------
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")

device = f"cuda:{CUDA_DEVICE}" if torch.cuda.is_available() and torch.cuda.device_count() > CUDA_DEVICE else "cpu"
print(f"Using device: {device}")
if device.startswith("cuda"):
    torch.cuda.set_device(CUDA_DEVICE)

# -------------------------
# LOAD MODEL + PIPELINE
# -------------------------
transcript_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    TRANSCRIPT_MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to(device)

transcript_processor = AutoProcessor.from_pretrained(TRANSCRIPT_MODEL_ID)

transcript_pipe = pipeline(
    "automatic-speech-recognition",
    model=transcript_model,
    tokenizer=transcript_processor.tokenizer,
    feature_extractor=transcript_processor.feature_extractor,
    torch_dtype=TORCH_DTYPE,
    device=device,  # HF pipeline accepts int device index too, but your string works in many setups
)

# -------------------------
# HELPERS
# -------------------------
def srt_timestamp(seconds: float) -> str:
    if seconds is None or seconds < 0:
        seconds = 0.0
    ms = int(round((seconds - int(seconds)) * 1000))
    total = int(seconds)
    hh = total // 3600
    mm = (total % 3600) // 60
    ss = total % 60
    return f"{hh:02d}:{mm:02d}:{ss:02d},{ms:03d}"

def clean_caption_text(t: str) -> str:
    t = (t or "").strip()
    t = re.sub(r"\s+", " ", t)
    return t

def enforce_max_chars(text: str, max_chars: int = 32) -> str:
    text = clean_caption_text(text)
    if len(text) <= max_chars:
        return text
    words = text.split(" ")
    lines, cur = [], ""
    for w in words:
        if len(cur) + len(w) + (1 if cur else 0) <= max_chars:
            cur = (cur + " " + w).strip()
        else:
            lines.append(cur)
            cur = w
    if cur:
        lines.append(cur)
    # 2 lines max for IG readability
    return "\n".join(lines[:2])

def transcribe_chunk(audio_data: np.ndarray, sample_rate: int, time_offset: float):
    """Return sentence-level chunks: [text, start_s, end_s_or_None]"""
    audio_input = {"raw": audio_data, "sampling_rate": sample_rate}
    kwargs = {"return_timestamps": True}
    if LANGUAGE:
        # many HF whisper pipelines accept "language" in generate_kwargs
        kwargs["generate_kwargs"] = {"language": LANGUAGE}

    out = transcript_pipe(audio_input, **kwargs)

    chunks = []
    for ch in out.get("chunks", []):
        text = (ch.get("text") or "").strip()
        ts = ch.get("timestamp", (None, None))
        if not (isinstance(ts, (list, tuple)) and len(ts) == 2):
            continue
        start, end = ts
        if start is None:
            continue
        start = float(start) + float(time_offset)
        end = (float(end) + float(time_offset)) if end is not None else None
        chunks.append([text, start, end])
    return chunks

def transcribe_long_audio(audio_data: np.ndarray, sample_rate: int, chunk_seconds: int = 300):
    full = []
    chunk_samples = int(chunk_seconds * sample_rate)
    n = len(audio_data)
    for i in range(0, n, chunk_samples):
        idx = i // chunk_samples + 1
        print(f"Processing chunk {idx} ({i/sample_rate:.1f}s -> {min(n, i+chunk_samples)/sample_rate:.1f}s)")
        chunk = audio_data[i:i+chunk_samples]
        offset = (i / sample_rate)
        full.extend(transcribe_chunk(chunk, sample_rate, offset))
    # sort by start time
    full.sort(key=lambda x: x[1])
    return full

def write_srt(chunks, out_path: str, default_dur=1.2, min_gap=0.04, max_chars=32):
    # filter empties
    chunks = [c for c in chunks if c and c[0] and c[1] is not None]
    chunks.sort(key=lambda x: x[1])

    with open(out_path, "w", encoding="utf-8") as f:
        out_index = 1
        for i, (text, start, end) in enumerate(chunks):
            # infer end if missing
            if end is None:
                if i + 1 < len(chunks):
                    next_start = chunks[i + 1][1]
                    end = max(start + 0.2, next_start - min_gap)
                else:
                    end = start + default_dur

            if end <= start:
                end = start + default_dur

            text = enforce_max_chars(text, max_chars=max_chars)
            if not text:
                continue

            f.write(f"{out_index}\n")
            f.write(f"{srt_timestamp(start)} --> {srt_timestamp(end)}\n")
            f.write(f"{text}\n\n")
            out_index += 1

    return out_path

# -------------------------
# LOAD AUDIO (.wav) + NORMALIZE
# -------------------------
wav_path = f"{FILE_STEM}.wav"
sr, audio = wavfile.read(wav_path)

# normalize to float32 mono
if audio.dtype == np.int16:
    audio = audio.astype(np.float32) / 32768.0
elif audio.dtype == np.int32:
    audio = audio.astype(np.float32) / 2147483648.0
else:
    audio = audio.astype(np.float32)

if audio.ndim > 1:
    audio = audio.mean(axis=1)

print(f"Loaded {wav_path}: sr={sr}, seconds={len(audio)/sr:.2f}")

# -------------------------
# TRANSCRIBE + WRITE SRT
# -------------------------
chunks = transcribe_long_audio(audio, sr, chunk_seconds=CHUNK_SECONDS)

srt_path = f"{FILE_STEM}.srt"
write_srt(chunks, srt_path, default_dur=DEFAULT_DUR, min_gap=MIN_GAP, max_chars=MAX_CHARS_PER_LINE)

# optional transcript text
txt_path = f"{FILE_STEM}.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    for t, s, e in chunks:
        f.write(clean_caption_text(t) + "\n")

print(f"✅ Wrote: {srt_path}")
print(f"✅ Wrote: {txt_path}")

CUDA available: True
CUDA devices: 2
Device 0: NVIDIA GeForce RTX 4090
Device 1: NVIDIA GeForce RTX 4060
Using device: cuda:0


Device set to use cuda:0
/home/pavlo/miniconda3/envs/speech_env/lib/python3.9/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


Loaded voice-1.wav: sr=48000, seconds=80.11
Processing chunk 1 (0.0s -> 80.1s)
✅ Wrote: voice-1.srt
✅ Wrote: voice-1.txt
